In [ ]:
import os
from getpass import getpass

kaggle_token = getpass("Paste Kaggle API here")

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(kaggle_token.strip())
os.chmod('/root/.kaggle/access_token', 0o600)

print("Done")

## Step 2: Libraries Install & Import

In [ ]:
!pip install scikit-surprise -q
!pip install kaggle -q

In [ ]:
import pandas as pd
import numpy as np
import ast
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel

from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
print("Libraries loaded")

## Step 3: Datasets Download



In [ ]:
# TMDB 5000 Movie Dataset
!kaggle datasets download -d tmdb/tmdb-movie-metadata -p /content/data --unzip

# MovieLens Small Dataset (ratings + movieId<->tmdbId mapping ke liye links.csv)
!kaggle datasets download -d shubhammehta21/movie-lens-small-latest-dataset -p /content/data/movielens --unzip

print("Dataset Download Complete")
!ls /content/data
!ls /content/data/movielens

## Step 4: Data Load

In [ ]:
movies = pd.read_csv('/content/data/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/data/tmdb_5000_credits.csv')

ratings = pd.read_csv('/content/data/movielens/ratings.csv')
links = pd.read_csv('/content/data/movielens/links.csv')
print("Movies:", movies.shape)
print("Credits:", credits.shape)
print("Ratings:", ratings.shape)
print("Links:", links.shape)

movies.head()

## Step 5: Content-Based Filtering




In [ ]:
credits.columns = ['id', 'title', 'cast', 'crew']
movies = movies.merge(credits, on='id')
movies = movies[['id', 'title_x', 'overview', 'genres', 'keywords', 'cast', 'crew', 'vote_average', 'vote_count']]
movies.rename(columns={'title_x': 'title'}, inplace=True)
movies.dropna(subset=['overview'], inplace=True)
movies.reset_index(drop=True, inplace=True)
movies.head()

In [ ]:
def get_names(text, limit=None):
    try:
        items = ast.literal_eval(text)
        names = [i['name'] for i in items]
        return names[:limit] if limit else names
    except Exception:
        return []

def get_director(text):
    try:
        crew = ast.literal_eval(text)
        for member in crew:
            if member['job'] == 'Director':
                return [member['name']]
        return []
    except Exception:
        return []

movies['genres_list'] = movies['genres'].apply(lambda x: get_names(x))
movies['keywords_list'] = movies['keywords'].apply(lambda x: get_names(x))
movies['cast_list'] = movies['cast'].apply(lambda x: get_names(x, limit=3))   
movies['director_list'] = movies['crew'].apply(get_director)

# Spaces hatayein taake 
def clean(lst):
    return [str(i).replace(" ", "").lower() for i in lst]

movies['genres_clean'] = movies['genres_list'].apply(clean)
movies['keywords_clean'] = movies['keywords_list'].apply(clean)
movies['cast_clean'] = movies['cast_list'].apply(clean)
movies['director_clean'] = movies['director_list'].apply(clean)

movies['overview_tokens'] = movies['overview'].apply(lambda x: str(x).lower().split())


movies['tags'] = (movies['overview_tokens'] + movies['genres_clean'] +
                   movies['keywords_clean'] + movies['cast_clean'] +
                   movies['director_clean'] * 2)   # director ko zyada weight

movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))
movies[['title', 'tags']].head()

In [ ]:
# TF-IDF vectorization + Cosine Similarity
tfidf = TfidfVectorizer(max_features=8000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['tags'])

content_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Content similarity matrix shape:", content_sim.shape)

# Title -> index lookup
indices = pd.Series(movies.index, index=movies['title'].str.lower()).drop_duplicates()

def get_content_recommendations(title, n=10):
    title = title.lower()
    if title not in indices:
        return f"'{title}' dataset not found"
    idx = indices[title]
    sim_scores = list(enumerate(content_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:n+1]
    movie_indices = [i[0] for i in sim_scores]
    return movies[['title', 'vote_average']].iloc[movie_indices]

get_content_recommendations("Avatar", 10)

## Step 6: Collaborative Filtering (SVD)



In [ ]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

svd = SVD(n_factors=50, n_epochs=20, random_state=42)

# Quick cross-validation check
cv_results = cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=3, verbose=True)

In [ ]:
trainset = data.build_full_trainset()
svd.fit(trainset)
print("Collaborative model (SVD) train Done")

def predict_rating(user_id, movie_id):
    return svd.predict(user_id, movie_id).est


print(predict_rating(user_id=1, movie_id=1))

## Step 7: Hybrid Recommender — Combine Both


In [ ]:
# movieId (MovieLens) <-> tmdbId (TMDB) mapping 
links_map = links.dropna(subset=['tmdbId'])[['movieId', 'tmdbId']]
links_map['tmdbId'] = links_map['tmdbId'].astype(int)

tmdb_to_movielens = dict(zip(links_map['tmdbId'], links_map['movieId']))
movielens_to_tmdb = dict(zip(links_map['movieId'], links_map['tmdbId']))

def hybrid_recommend(user_id, title, n=10, content_weight=0.5, collab_weight=0.5):
    title = title.lower()
    if title not in indices:
        return f"'{title}' dataset not found"

    idx = indices[title]
    sim_scores = list(enumerate(content_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:50]  # top 50 candidates

    results = []
    for i, content_score in sim_scores:
        tmdb_id = movies.iloc[i]['id']
        movielens_id = tmdb_to_movielens.get(tmdb_id)

        if movielens_id is not None:
            collab_score = predict_rating(user_id, movielens_id) / 5.0   # normalize 0-1
        else:
            collab_score = movies.iloc[i]['vote_average'] / 10.0          # fallback

        hybrid_score = (content_weight * content_score) + (collab_weight * collab_score)
        results.append((movies.iloc[i]['title'], hybrid_score))

    results = sorted(results, key=lambda x: x[1], reverse=True)[:n]
    return pd.DataFrame(results, columns=['title', 'hybrid_score'])


hybrid_recommend(user_id=1, title="Avatar", n=10)

## Step 8: Models Save

In [ ]:
import pickle

with open('/content/movies.pkl', 'wb') as f:
    pickle.dump(movies, f)

with open('/content/content_sim.pkl', 'wb') as f:
    pickle.dump(content_sim, f)

with open('/content/svd_model.pkl', 'wb') as f:
    pickle.dump(svd, f)

with open('/content/links_map.pkl', 'wb') as f:
    pickle.dump({'tmdb_to_movielens': tmdb_to_movielens}, f)

print("Models saved !")
print("Use in WebApp")

## Step 9: TMDB API Key (For showing the Movie Posters)


In [ ]:
from getpass import getpass

tmdb_api_key = getpass("Paste TMDB API here")

with open('/content/tmdb_api_key.txt', 'w') as f:
    f.write(tmdb_api_key.strip())

print("TMDB API key Saved !")

## Step 10: HTML + CSS + JavaScript Interface


In [ ]:
import os
os.makedirs('/content/templates', exist_ok=True)
os.makedirs('/content/static', exist_ok=True)
print("Folders ready ")

### `templates/index.html` — page structure

In [ ]:
%%writefile /content/templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <title>Movie Recommender</title>
  <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}" />
</head>
<body>
  <header class="header">
    <h1>🎬 Movie Recommender</h1>
    <p>Content-based + Collaborative Filtering movie recommendations</p>
  </header>

  <section class="controls">
    <div class="control-group">
      <label for="movieInput">Movie chunein</label>
      <input list="movieList" id="movieInput" placeholder="Type Movie Name..." />
      <datalist id="movieList"></datalist>
    </div>

    <div class="control-group">
      <label for="userId">User ID</label>
      <input type="number" id="userId" min="1" value="1" />
    </div>

    <div class="control-group">
      <label for="countInput">Number of recommendations?</label>
      <input type="number" id="countInput" min="5" max="20" value="10" />
    </div>

    <div class="control-group slider-group">
      <label for="weightInput">Content ↔ Collaborative weight (<span id="weightValue">0.5</span>)</label>
      <input type="range" id="weightInput" min="0" max="1" step="0.1" value="0.5" />
    </div>

    <button id="recommendBtn">Recommend</button>
  </section>

  <section id="status"></section>

  <section id="results" class="grid"></section>

  <script src="{{ url_for('static', filename='script.js') }}"></script>
</body>
</html>

### `static/style.css` — styling

In [ ]:
%%writefile /content/static/style.css
* { box-sizing: border-box; }

body {
  margin: 0;
  font-family: 'Segoe UI', Roboto, Arial, sans-serif;
  background: linear-gradient(180deg, #f7f8fc 0%, #eef1fb 100%);
  color: #2b2d42;
  min-height: 100vh;
}

.header {
  text-align: center;
  padding: 48px 20px 24px;
}

.header h1 {
  margin: 0 0 8px;
  font-size: 2.3rem;
  font-weight: 800;
  background: linear-gradient(90deg, #6c5ce7, #ff6b81);
  -webkit-background-clip: text;
  background-clip: text;
  color: transparent;
}

.header p {
  color: #6b7089;
  margin: 0;
  font-size: 1rem;
}

.controls {
  max-width: 900px;
  margin: 20px auto;
  padding: 26px;
  background: #ffffff;
  border-radius: 18px;
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
  gap: 18px;
  align-items: end;
  box-shadow: 0 10px 30px rgba(108, 92, 231, 0.10);
  border: 1px solid #eceef8;
}

.control-group {
  display: flex;
  flex-direction: column;
  gap: 6px;
}

.slider-group {
  grid-column: 1 / -1;
}

label {
  font-size: 0.85rem;
  font-weight: 600;
  color: #4a4d68;
}

input[type="text"],
input[type="number"],
input {
  padding: 11px 14px;
  border-radius: 10px;
  border: 1.5px solid #e1e3f0;
  background: #f9fafe;
  color: #2b2d42;
  font-size: 0.95rem;
  outline: none;
  transition: border-color 0.15s ease, box-shadow 0.15s ease;
}

input[type="text"]:focus,
input[type="number"]:focus {
  border-color: #6c5ce7;
  box-shadow: 0 0 0 3px rgba(108, 92, 231, 0.15);
}

input[type="range"] {
  padding: 0;
  accent-color: #6c5ce7;
}

button {
  grid-column: 1 / -1;
  padding: 15px;
  border: none;
  border-radius: 12px;
  background: linear-gradient(90deg, #6c5ce7, #ff6b81);
  color: #ffffff;
  font-weight: 700;
  font-size: 1rem;
  cursor: pointer;
  box-shadow: 0 8px 20px rgba(108, 92, 231, 0.30);
  transition: transform 0.15s ease, box-shadow 0.15s ease, opacity 0.15s ease;
}

button:hover { transform: translateY(-2px); box-shadow: 0 12px 24px rgba(108, 92, 231, 0.38); }
button:active { transform: translateY(0); }
button:disabled { opacity: 0.6; cursor: not-allowed; box-shadow: none; }

#status {
  text-align: center;
  min-height: 24px;
  color: #6c5ce7;
  font-weight: 600;
  margin: 16px 0;
}

.grid {
  max-width: 1200px;
  margin: 20px auto 60px;
  padding: 0 20px;
  display: grid;
  grid-template-columns: repeat(auto-fill, minmax(190px, 1fr));
  gap: 24px;
}

.card {
  background: #ffffff;
  border-radius: 16px;
  overflow: hidden;
  box-shadow: 0 6px 18px rgba(43, 45, 66, 0.08);
  border: 1px solid #eceef8;
  transition: transform 0.2s ease, box-shadow 0.2s ease;
}

.card:hover {
  transform: translateY(-6px);
  box-shadow: 0 16px 32px rgba(108, 92, 231, 0.18);
}

.card img {
  width: 100%;
  height: 270px;
  object-fit: cover;
  display: block;
  background: #eceef8;
}

.card-body {
  padding: 14px 16px 18px;
}

.card-title {
  font-size: 0.95rem;
  font-weight: 700;
  margin: 0 0 8px;
  line-height: 1.35;
  color: #2b2d42;
}

.card-meta {
  display: flex;
  justify-content: space-between;
  align-items: center;
  font-size: 0.8rem;
  color: #6b7089;
}

.badge {
  background: rgba(108, 92, 231, 0.12);
  color: #6c5ce7;
  font-weight: 700;
  padding: 3px 10px;
  border-radius: 20px;
}

### `static/script.js` — interactivity

In [ ]:
%%writefile /content/static/script.js
const movieInput = document.getElementById('movieInput');
const movieList = document.getElementById('movieList');
const userIdInput = document.getElementById('userId');
const countInput = document.getElementById('countInput');
const weightInput = document.getElementById('weightInput');
const weightValue = document.getElementById('weightValue');
const recommendBtn = document.getElementById('recommendBtn');
const statusEl = document.getElementById('status');
const resultsEl = document.getElementById('results');

weightInput.addEventListener('input', () => {
  weightValue.textContent = weightInput.value;
});

fetch('/api/movies')
  .then(res => res.json())
  .then(movies => {
    movies.forEach(title => {
      const option = document.createElement('option');
      option.value = title;
      movieList.appendChild(option);
    });
  })
  .catch(() => {
    statusEl.textContent = 'Movie list Cannot Load.';
  });

recommendBtn.addEventListener('click', async () => {
  const movie = movieInput.value.trim();
  if (!movie) {
    statusEl.textContent = 'Type one movie name';
    return;
  }

  const userId = userIdInput.value || 1;
  const n = countInput.value || 10;
  const weight = weightInput.value || 0.5;

  resultsEl.innerHTML = '';
  statusEl.textContent = 'Recommendations...';
  recommendBtn.disabled = true;

  try {
    const params = new URLSearchParams({ movie, user_id: userId, n, weight });
    const res = await fetch(`/api/recommend?${params.toString()}`);
    const data = await res.json();

    if (!data || data.length === 0) {
      statusEl.textContent = `'${movie}' Type Correct Name`;
      return;
    }

    statusEl.textContent = '';
    data.forEach(movieItem => {
      const card = document.createElement('div');
      card.className = 'card';
      card.innerHTML = `
        <img src="${movieItem.poster}" alt="${movieItem.title}" onerror="this.src='https://via.placeholder.com/500x750?text=No+Image'" />
        <div class="card-body">
          <p class="card-title">${movieItem.title}</p>
          <div class="card-meta">
            <span>⭐ ${movieItem.rating.toFixed(1)}</span>
            <span class="badge">score ${movieItem.score}</span>
          </div>
        </div>
      `;
      resultsEl.appendChild(card);
    });
  } catch (err) {
    statusEl.textContent = 'Try Again';
  } finally {
    recommendBtn.disabled = false;
  }
});

### `app.py` — Flask backend (hybrid logic + poster fetching + API endpoints)

In [ ]:
%%writefile /content/app.py
from flask import Flask, jsonify, request, render_template
import pickle
import pandas as pd
import requests

app = Flask(__name__, template_folder="templates", static_folder="static")

movies = pickle.load(open("movies.pkl", "rb"))
content_sim = pickle.load(open("content_sim.pkl", "rb"))
svd = pickle.load(open("svd_model.pkl", "rb"))
links_data = pickle.load(open("links_map.pkl", "rb"))
tmdb_to_movielens = links_data["tmdb_to_movielens"]

indices = pd.Series(movies.index, index=movies["title"].str.lower()).drop_duplicates()

with open("tmdb_api_key.txt") as f:
    TMDB_API_KEY = f.read().strip()

poster_cache = {}

def fetch_poster(tmdb_id):
    if tmdb_id in poster_cache:
        return poster_cache[tmdb_id]
    fallback = "https://via.placeholder.com/500x750?text=No+Image"
    try:
        url = f"https://api.themoviedb.org/3/movie/{int(tmdb_id)}"
        res = requests.get(url, params={"api_key": TMDB_API_KEY, "language": "en-US"}, timeout=5)
        data = res.json()
        poster_path = data.get("poster_path")
        full_url = f"https://image.tmdb.org/t/p/w500{poster_path}" if poster_path else fallback
    except Exception:
        full_url = fallback
    poster_cache[tmdb_id] = full_url
    return full_url

def hybrid_recommend(user_id, title, n=10, content_weight=0.5, collab_weight=0.5):
    title = title.lower()
    if title not in indices:
        return []

    idx = indices[title]
    sim_scores = list(enumerate(content_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:50]

    results = []
    for i, content_score in sim_scores:
        tmdb_id = movies.iloc[i]["id"]
        movielens_id = tmdb_to_movielens.get(tmdb_id)
        if movielens_id is not None:
            collab_score = svd.predict(user_id, movielens_id).est / 5.0
        else:
            collab_score = movies.iloc[i]["vote_average"] / 10.0

        hybrid_score = (content_weight * content_score) + (collab_weight * collab_score)
        results.append({
            "title": movies.iloc[i]["title"],
            "tmdb_id": int(tmdb_id),
            "rating": float(movies.iloc[i]["vote_average"]),
            "score": round(float(hybrid_score), 3),
        })

    results = sorted(results, key=lambda x: x["score"], reverse=True)[:n]
    for r in results:
        r["poster"] = fetch_poster(r["tmdb_id"])
    return results

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/api/movies")
def api_movies():
    return jsonify(sorted(movies["title"].unique().tolist()))

@app.route("/api/recommend")
def api_recommend():
    title = request.args.get("movie", "")
    user_id = int(request.args.get("user_id", 1))
    n = int(request.args.get("n", 10))
    weight = float(request.args.get("weight", 0.5))
    recs = hybrid_recommend(user_id, title, n=n, content_weight=weight, collab_weight=1 - weight)
    return jsonify(recs)

if __name__ == "__main__":
    app.run(port=5000)

## Step 11: Run app with ngrok


In [ ]:
!pip install flask pyngrok requests -q

from getpass import getpass
from pyngrok import ngrok

ngrok_token = getpass("Paste ngrok Token here")
ngrok.set_auth_token(ngrok_token.strip())
print("ngrok authtoken Done !")

In [ ]:
import subprocess, time

ngrok.kill()

# Flask app ko background mein start karein
flask_process = subprocess.Popen(["python", "/content/app.py"], cwd="/content")
time.sleep(4)   # Flask ko start hone ka time dein

public_url = ngrok.connect(5000)
print("Interface:")
print(public_url)

In [ ]:
# App band karne ke liye ye cell run karein
ngrok.kill()
flask_process.terminate()
print("APP Closed")